# Context and thinking tokens in a small LLM

A support desk receives thousands of messages a day. Each message has to reach the right team. In this assignment you turn Qwen3-0.6B, a small instruction-tuned language model, into a router. It reads a customer message and answers with one of 27 intents in JSON. You measure how many tokens it uses for each message.

The first part is about the tokens that go into the model. You show it labeled examples before each message. You measure how accuracy, prompt length, time and memory change as the number of examples grows from none to hundreds. You then fine-tune the model with LoRA so that it needs no examples. You compare prompting with fine-tuning. Finally, you hide the message inside a long inbox of other messages. For the base model and the fine-tuned one, you measure accuracy when the message is at the start, in the middle and at the end.

The second part is about the tokens that come out before the answer. Qwen3 can write its reasoning in a `<think>` block and then answer. You measure whether reasoning helps on this classification task and how many generated tokens and how much time it uses. You also test the router you fine-tuned in the first part, whose training data contained no reasoning.

Every `TODO` starts as a stub that runs. Before you change anything, all cells run from top to bottom and produce the baseline. The baseline uses no few-shot examples, a single message instead of an inbox, and an answer with the reasoning still attached. The questions at the end of each part are graded on what you measure and how you explain it.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "transformers>=4.51" "trl>=0.19" peft datasets accelerate

In [ ]:
import os
import warnings

import datasets
import transformers

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")
transformers.utils.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
datasets.disable_progress_bars()

In [ ]:
import json
import math
import random
import re
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import Dataset, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"[INFO] device: {device}, bfloat16: {BF16}")
if device == "cuda":
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")

## Settings

Keep the defaults for the graded run.

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B"
SEED = 0
TEST_PER_INTENT = 10  # 270 test messages
torch.manual_seed(SEED)

## The data

The [Bitext customer support data set](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) has 26,872 messages that customers might send to a support desk. Each message has one of 27 *intent* labels, such as `cancel_order` or `track_refund`. The intents fall into 11 categories. The messages are synthetic and use placeholders such as `{{Order Number}}`. The `flags` column marks how each message is phrased, for example `B` for basic syntax and `Z` for typos.

The cell holds out 10 messages per intent, 270 in all, as the test set. Every measurement in this assignment uses this test set or a fixed part of it. The remaining messages are the pool you draw training examples and few-shot examples from.

In [ ]:
raw = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
INTENTS = sorted(set(raw["intent"]))
print(f"[INFO] {len(raw):,} messages, {len(INTENTS)} intents")

rng = random.Random(SEED)
by_intent = defaultdict(list)
for row in raw:
    by_intent[row["intent"]].append({"text": row["instruction"], "intent": row["intent"], "flags": row["flags"]})
TEST, POOL = [], defaultdict(list)
for intent in INTENTS:
    rows = by_intent[intent][:]
    rng.shuffle(rows)
    TEST += rows[:TEST_PER_INTENT]
    POOL[intent] = rows[TEST_PER_INTENT:]
print(f"[INFO] test set: {len(TEST)} messages; pool: {sum(len(v) for v in POOL.values()):,} messages")
print(f"[INFO] example: {TEST[0]['text']!r} -> {TEST[0]['intent']}")

## The model and the router prompt

Qwen3-0.6B is a 0.6-billion-parameter instruction-tuned model with a context window of 32,768 tokens. The router prompt puts the list of allowed intents in the system message and the customer message in the user turn. It asks the model to return JSON. Qwen3 can reason in a `<think>` block before it answers; `enable_thinking=False` turns that off, and this assignment keeps it off.

`route` generates answers for a batch of prompts. `score` parses the answers. An answer counts as valid when it contains a JSON object with an `intent` field, and as correct when that field is the right intent.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"  # generation continues from the right end of each prompt


def load_model(path=MODEL_ID):
    m = AutoModelForCausalLM.from_pretrained(path, dtype=torch.bfloat16 if BF16 else torch.float16)
    return m.to(device).eval()


def system_prompt(labels):
    return ("You route customer support messages. Choose exactly one intent from this list:\n"
            + ", ".join(labels)
            + '\nAnswer with JSON only, in the form {"intent": "<intent>"}.')


def build_prompt(text, labels, history=()):
    """Chat-formatted prompt text. history is a list of (message, label) pairs shown before it."""
    messages = [{"role": "system", "content": system_prompt(labels)}]
    for msg, label in history:
        messages += [{"role": "user", "content": msg},
                     {"role": "assistant", "content": json.dumps({"intent": label})}]
    messages.append({"role": "user", "content": text})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)


@torch.no_grad()
def route(model, prompts, batch_size=None, max_new_tokens=24):
    """Generate answers; return the texts, the prompt lengths in tokens and the seconds per prompt.

    The batch size shrinks as the prompts grow, so that the key-value cache of a batch stays
    near 40,000 tokens (about 4.5 GB for this model).
    """
    if batch_size is None:
        longest = max(len(tokenizer(p)["input_ids"]) for p in prompts[:8])
        batch_size = max(1, min(32, 40_000 // longest))
    texts, lengths = [], []
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for i in range(0, len(prompts), batch_size):
        batch = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(device)
        out = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        texts += tokenizer.batch_decode(out[:, batch["input_ids"].shape[1]:], skip_special_tokens=True)
        lengths += batch["attention_mask"].sum(1).tolist()
    if device == "cuda":
        torch.cuda.synchronize()
    return texts, lengths, (time.perf_counter() - start) / len(prompts)


def parse_intent(text):
    match = re.search(r"\{.*?\}", text, re.S)
    if not match:
        return None
    try:
        return json.loads(match.group(0)).get("intent")
    except (json.JSONDecodeError, AttributeError):
        return None


def score(texts, truth):
    parsed = [parse_intent(t) for t in texts]
    return {"accuracy": float(np.mean([p == t for p, t in zip(parsed, truth)])),
            "valid_json": float(np.mean([p is not None for p in parsed]))}

Use the base model with no examples as the baseline.

In [ ]:
base = load_model()
truth = [r["intent"] for r in TEST]
texts, lengths, sec = route(base, [build_prompt(r["text"], INTENTS) for r in TEST])
zero_shot = score(texts, truth)
print(f"[INFO] zero-shot: accuracy {zero_shot['accuracy']:.3f}, valid JSON {zero_shot['valid_json']:.3f}, "
      f"{np.mean(lengths):.0f} prompt tokens, {1000 * sec:.1f} ms per message")
for t, r in list(zip(texts, TEST))[:3]:
    print(f"  {r['text'][:60]!r:<64} truth: {r['intent']:<22} answer: {t.strip()!r}")

## Part 1: context tokens

### Few-shot examples

A language model can learn a task from examples placed in its prompt, without any change to its weights (*in-context learning*). Each example is a customer message followed by the correct JSON answer, as a previous turn of the conversation. Every example also lengthens the prompt. The model processes the whole prompt for every message it routes.

**TODO 1.** Write `few_shot_history`, which returns `k_per_intent` labeled examples for every intent, drawn from `POOL` (never from `TEST`), as a list of `(message, intent)` pairs in random order. The stub returns no examples.

In [ ]:
def few_shot_history(k_per_intent, seed=SEED):
    """Return k_per_intent (message, intent) pairs per intent from POOL, in random order."""
    # TODO 1: draw the examples from POOL and shuffle them
    return []

The cell routes the 270 test messages with 0, 1, 4 and 16 examples per intent, that is up to 432 examples in the prompt, and records the accuracy, the prompt length, the time per message and the peak GPU memory.

In [ ]:
FEW_SHOT = []
for k in (0, 1, 4, 16):
    history = few_shot_history(k)
    prompts = [build_prompt(r["text"], INTENTS, history) for r in TEST]
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    texts, lengths, sec = route(base, prompts)
    peak = torch.cuda.max_memory_allocated() / 2**30 if device == "cuda" else float("nan")
    FEW_SHOT.append({"k": k, **score(texts, truth), "tokens": float(np.mean(lengths)),
                     "ms": 1000 * sec, "peak_gb": peak})
    print(f"k = {k:>2} per intent: accuracy {FEW_SHOT[-1]['accuracy']:.3f}, "
          f"{FEW_SHOT[-1]['tokens']:>7.0f} prompt tokens, {1000 * sec:>7.1f} ms per message, "
          f"peak memory {peak:.2f} GB")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
tokens = [r["tokens"] for r in FEW_SHOT]
for ax, key, label in zip(axes, ("accuracy", "ms", "peak_gb"),
                          ("test accuracy", "ms per message", "peak GPU memory (GB)")):
    ax.plot(tokens, [r[key] for r in FEW_SHOT], "o-")
    for r in FEW_SHOT:
        ax.annotate(f"k={r['k']}", (r["tokens"], r[key]), textcoords="offset points", xytext=(4, 4), fontsize=8)
    ax.set_xscale("log")
    ax.set_xlabel("prompt tokens")
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Fine-tuning

Fine-tuning updates the model's weights using the examples. The cell trains LoRA adapters on 32 examples per intent (864 messages, 2 epochs) and routes the test set with no examples in the prompt.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


def train_router(rows, label_map, lora_rank=16, full=False, epochs=2, lr=None, out="sft-run"):
    """Fine-tune the router on rows, whose intents are written with label_map.

    Each example is a prompt (system message and customer message) and a completion (the JSON
    answer); the loss is computed on the completion only. With full=False only LoRA adapters of
    rank lora_rank are trained; with full=True every weight is.
    """
    labels = sorted(set(label_map.values()))
    data = Dataset.from_list([
        {"prompt": build_prompt(r["text"], labels),
         "completion": json.dumps({"intent": label_map[r["intent"]]}) + tokenizer.eos_token}
        for r in rows
    ])
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.bfloat16 if BF16 else torch.float32)
    # warmup_steps below 1 is a fraction of the training steps
    args = SFTConfig(
        output_dir=out, num_train_epochs=epochs, per_device_train_batch_size=8,
        gradient_accumulation_steps=2, learning_rate=lr or (2e-5 if full else 2e-4),
        lr_scheduler_type="cosine", warmup_steps=0.05, logging_steps=20, save_strategy="no",
        report_to="none", bf16=BF16, fp16=not BF16 and device == "cuda",
        gradient_checkpointing=full, completion_only_loss=True, disable_tqdm=True,
    )
    peft = None if full else LoraConfig(r=lora_rank, lora_alpha=2 * lora_rank, lora_dropout=0.05,
                                        target_modules="all-linear", task_type="CAUSAL_LM")
    trainer = SFTTrainer(model=model, args=args, train_dataset=data, peft_config=peft,
                         processing_class=tokenizer)
    start = time.perf_counter()
    trainer.train()
    model = trainer.model.merge_and_unload() if not full else trainer.model
    model = model.to(torch.bfloat16 if BF16 else torch.float16).eval()
    print(f"[INFO] trained on {len(rows):,} examples in {time.perf_counter() - start:.0f} s")
    return model


def sample_rows(per_intent, seed=SEED):
    r = random.Random(seed)
    return [row for intent in INTENTS for row in r.sample(POOL[intent], per_intent)]


REAL = {i: i for i in INTENTS}
del base
torch.cuda.empty_cache()
tuned = train_router(sample_rows(32), REAL)
tuned.save_pretrained("router-lora")  # used again in Part 2
texts, lengths, sec = route(tuned, [build_prompt(r["text"], INTENTS) for r in TEST])
SFT_RESULT = {**score(texts, truth), "tokens": float(np.mean(lengths)), "ms": 1000 * sec}
print(f"[INFO] fine-tuned, no examples: accuracy {SFT_RESULT['accuracy']:.3f}, "
      f"{SFT_RESULT['tokens']:.0f} prompt tokens, {SFT_RESULT['ms']:.1f} ms per message")
del tuned
torch.cuda.empty_cache()

### Position in a long context

A model with a 32,768-token window does not necessarily use every position in it equally well. Liu et al. found that language models answer best when the relevant information is at the start or the end of a long context, and worst when it is in the middle ([Lost in the Middle](https://arxiv.org/abs/2307.03172)). You test this on the base model and on the router you just fine-tuned. The router routes single messages almost perfectly, so a drop in its accuracy inside the inbox shows the effect of the long context. Instead of a single message, each model receives an inbox of messages, one of which is marked as the message to route.

**TODO 2.** Write `inbox_prompt`. It builds an inbox of `n_messages` messages, one per line, in which the message to route is placed at the relative `position` (0 for the first line, 0.5 for the middle, 1 for the last) and starts with the marker `[ROUTE THIS]`. The other messages are drawn from `DISTRACTORS`. The user turn is the inbox followed by `Route the message marked [ROUTE THIS].` The stub sends the target message alone.

In [ ]:
DISTRACTORS = [row["text"] for intent in INTENTS for row in POOL[intent][-40:]]


def inbox_prompt(target_text, n_messages, position, seed=SEED):
    """Return the router prompt for an inbox of n_messages with the marked target at the given position."""
    # TODO 2: build the inbox with the marked target at the given position
    return build_prompt(target_text, INTENTS)

The cell routes 81 test messages (3 per intent) with each model, inside inboxes of 25, 100 and 400 messages, with the marked message at the start, in the middle and at the end, and prints the accuracy of each setting.

In [ ]:
SUBSET = [r for intent in INTENTS for r in [x for x in TEST if x["intent"] == intent][:3]]
sub_truth = [r["intent"] for r in SUBSET]
SIZES, POSITIONS = (25, 100, 400), (0.0, 0.5, 1.0)
POSITION_ACC, POSITION_TOKENS = {}, {}
for name, path in [("base", MODEL_ID), ("fine-tuned router", "router-lora")]:
    model = load_model(path)
    single, _, _ = route(model, [build_prompt(r["text"], INTENTS) for r in SUBSET])
    POSITION_ACC[name] = np.zeros((len(SIZES), len(POSITIONS)))
    print(f"{name}: single message {score(single, sub_truth)['accuracy']:.3f}")
    for i, n in enumerate(SIZES):
        for j, pos in enumerate(POSITIONS):
            prompts = [inbox_prompt(r["text"], n, pos, seed=SEED + m) for m, r in enumerate(SUBSET)]
            texts, lengths, _ = route(model, prompts)
            POSITION_ACC[name][i, j] = score(texts, sub_truth)["accuracy"]
            POSITION_TOKENS[n] = float(np.mean(lengths))
        print(f"  {n:>4} messages ({POSITION_TOKENS[n]:>6.0f} tokens): "
              + "  ".join(f"position {p}: {a:.3f}" for p, a in zip(POSITIONS, POSITION_ACC[name][i])))
    del model
    torch.cuda.empty_cache()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (name, acc) in zip(axes, POSITION_ACC.items()):
    im = ax.imshow(acc, cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(POSITIONS)))
    ax.set_xticklabels(["start", "middle", "end"])
    ax.set_yticks(range(len(SIZES)))
    ax.set_yticklabels([f"{n} msgs\n({POSITION_TOKENS[n]:.0f} tok)" for n in SIZES])
    for i in range(len(SIZES)):
        for j in range(len(POSITIONS)):
            ax.text(j, i, f"{acc[i, j]:.2f}", ha="center", va="center",
                    color="black" if acc[i, j] > 0.5 else "white")
    ax.set_xlabel("position of the marked message")
    ax.set_title(name)
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

### Questions for Part 1

**Question 1.** How do accuracy, prompt length, time per message and peak memory change with the number of examples per intent? Which grows fastest, and why? Use your measured numbers.

*Your answer:*

**Question 2.** Does the position of the marked message matter, and does the effect change with the size of the inbox? Compare the base model with the fine-tuned router, compare your results with the U-shaped curve of Liu et al., and explain any difference. Remember that the router was trained on single messages, never on inboxes.

*Your answer:*

**Question 3.** The fine-tuned router uses a short prompt. Compare it with the best few-shot setting on accuracy and on cost per message. If the support desk routes one million messages a day, which would you deploy, and what would change your mind?

*Your answer:*

## Part 2: thinking tokens

With `enable_thinking=True`, Qwen3 first writes its reasoning between `<think>` and `</think>` and then the answer. Reasoning can help on problems that need several steps. It also uses more tokens. Each generated token adds GPU time and, behind an API, cost. The cell below generates with thinking on and off for 135 test messages (5 per intent), with the sampling settings that the Qwen3 model card recommends for each mode and a limit of 1,024 new tokens.

**TODO 3.** Write `split_thinking`, which takes a generated text and returns the number of tokens of reasoning (the text up to and including `</think>`, or the whole text if `</think>` never appears) and the answer that follows. The stub treats the whole text as the answer.

In [ ]:
def split_thinking(text):
    """Return (reasoning_tokens, answer_text) for one generated text."""
    # TODO 3: separate the <think> block from the answer and count its tokens
    return 0, text

In [ ]:
THINK_SUBSET = [r for intent in INTENTS for r in [x for x in TEST if x["intent"] == intent][:5]]
think_truth = [r["intent"] for r in THINK_SUBSET]


def think_prompt(text, thinking):
    messages = [{"role": "system", "content": system_prompt(INTENTS)}, {"role": "user", "content": text}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=thinking)


@torch.no_grad()
def generate_mode(model, thinking, batch_size=16, max_new_tokens=1024):
    torch.manual_seed(SEED)
    sampling = dict(temperature=0.6, top_p=0.95, top_k=20) if thinking else dict(temperature=0.7, top_p=0.8, top_k=20)
    prompts = [think_prompt(r["text"], thinking) for r in THINK_SUBSET]
    texts, new_tokens = [], []
    start = time.perf_counter()
    for i in range(0, len(prompts), batch_size):
        batch = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(device)
        out = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=True,
                             pad_token_id=tokenizer.pad_token_id, **sampling)
        gen = out[:, batch["input_ids"].shape[1]:]
        texts += tokenizer.batch_decode(gen, skip_special_tokens=True)
        new_tokens += (gen != tokenizer.pad_token_id).sum(1).tolist()
    sec = (time.perf_counter() - start) / len(prompts)
    reasoning, answers = zip(*[split_thinking(t) for t in texts])
    return {**score(list(answers), think_truth), "new_tokens": float(np.mean(new_tokens)),
            "reasoning_tokens": float(np.mean(reasoning)), "cut_off": float(np.mean([n >= max_new_tokens for n in new_tokens])),
            "ms": 1000 * sec, "example": texts[0]}


THINKING = {}
for label, path in [("base", MODEL_ID), ("fine-tuned router", "router-lora")]:
    model = load_model(path)
    for thinking in (False, True):
        r = generate_mode(model, thinking)
        THINKING[(label, thinking)] = r
        print(f"{label:<16} thinking={str(thinking):<5} accuracy {r['accuracy']:.3f}  valid JSON {r['valid_json']:.3f}  "
              f"new tokens {r['new_tokens']:>6.1f}  reasoning tokens {r['reasoning_tokens']:>6.1f}  "
              f"cut off {r['cut_off']:.2f}  {r['ms']:>7.1f} ms per message")
    del model
    torch.cuda.empty_cache()

In [ ]:
for key, r in THINKING.items():
    print(f"--- {key[0]}, thinking={key[1]}: first answer, shortened")
    print(r["example"][:600])

### Questions for Part 2

**Question 4.** For the base model, does thinking improve the routing accuracy? Compare the gain, if any, with the cost in generated tokens and time per message, and estimate the cost of routing one million messages a day in each mode.

*Your answer:*

**Question 5.** The router of Part 1 was fine-tuned on answers without reasoning. What happens when you turn thinking on for it? Explain from the training data and the chat template what the fine-tuning did to the model's reasoning behavior.

*Your answer:*

**Question 6.** Name a kind of support message, or a change to the task, for which you would expect reasoning to help. Explain why, and how you would test it.

*Your answer:*

## References

* Brown et al., [Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165), 2020.
* Hsieh et al., [RULER: What's the Real Context Size of Your Long-Context Language Models?](https://arxiv.org/abs/2404.06654), 2024.
* Hu et al., [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685), 2021.
* Liu et al., [Lost in the Middle: How Language Models Use Long Contexts](https://arxiv.org/abs/2307.03172), 2023.
* Mosbach et al., [Few-shot Fine-tuning vs. In-context Learning: A Fair Comparison and Evaluation](https://arxiv.org/abs/2305.16938), 2023.
* Qwen Team, [Qwen3 Technical Report](https://arxiv.org/abs/2505.09388), 2025.
* Sprague et al., [To CoT or Not to CoT? Chain-of-Thought Helps Mainly on Math and Symbolic Reasoning](https://arxiv.org/abs/2409.12183), 2024.
* Wei et al., [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903), 2022.